# OPTIMIZED PIPELINE 2: FULL DATASET (COST-SENSITIVE & CUSTOM LOSS)

In [9]:
import pandas as pd
import numpy as np
import os
from sklearn.preprocessing import RobustScaler

OUT_DIR = r"c:\Users\edgib\Downloads\PREDICTIVE MODEL USING NEURAL NETWORK\DATASETS AFTER MICE"

In [10]:
# 1. LOAD POST-MICE DATASETS (CSV FROM PIPELINE 1)
X_train = pd.read_csv(os.path.join(OUT_DIR, 'X_TRAIN_AFTER_CLIP.csv'))
X_test = pd.read_csv(os.path.join(OUT_DIR, 'X_TEST_AFTER_CLIP.csv'))
y_train = pd.read_csv(os.path.join(OUT_DIR, 'Y_TRAIN_AFTER_CLIP.csv'))['target']
y_test = pd.read_csv(os.path.join(OUT_DIR, 'Y_TEST_AFTER_CLIP.csv'))['target']

In [11]:
# NO UNDERSAMPLING APPLIED (FULL 80/20 IS PRESERVED)

# 2. TARGET ENCODING
# addr_state is read as string from CSV, which is exactly what we need here
state_means = y_train.groupby(X_train['addr_state']).mean()
X_train['addr_state'] = X_train['addr_state'].map(state_means)
X_test['addr_state'] = X_test['addr_state'].map(state_means)

# 3. SCALING
scaler = RobustScaler()
X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train), columns=X_train.columns)
X_test_scaled = pd.DataFrame(scaler.transform(X_test), columns=X_test.columns)

In [ ]:
# 4. CORRELATION FILTER
train_full = X_train_scaled.copy()
train_full['TARGET'] = y_train.values

corr_categories = X_train_scaled.corr().abs()
corr_target = train_full.corr().abs()['TARGET']
upper_tri = corr_categories.where(np.triu(np.ones(corr_categories.shape), k=1).astype(bool))

columns_to_eliminate = set()
for col in upper_tri.columns:
    for corr_col in upper_tri.index[upper_tri[col] > 0.9]:
        if corr_target[col] < corr_target[corr_col]:
            columns_to_eliminate.add(col)
        else:
            columns_to_eliminate.add(corr_col)

# checking if the eliminted columns are the same than in the
# undersampling approach. If they are, it means that the correlation filter is not affected by the elimination of undersampling, which would be a good sign in terms of consistency between pipelines.
columns_to_eliminate = list(columns_to_eliminate)


X_train_final = X_train_scaled.drop(columns=list(columns_to_eliminate) + ['purpose_educational'], errors='ignore')
X_test_final = X_test_scaled.drop(columns=list(columns_to_eliminate) + ['purpose_educational'], errors='ignore')

In [ ]:
# the eliminated columns are actually the same in both pipelines.
print(columns_to_eliminate)

['num_sats', 'tot_cur_bal', 'fico_range_high', 'installment', 'num_rev_tl_bal_gt_0', 'int_rate', 'mnths_since_earliest_cr']


In [ ]:
# checking if the shapes of the final datasets are consistent between pipelines. The most important thing is that the test set is exactly equal in both cases, which actually is. This ensures a proper economic comparison.

print(X_train_final.shape)
print(X_test_final.shape)

(960440, 72)
(240111, 72)


In [14]:
# 5. EXPORT FINAL DATASETS (MODELS 2 AND 3)
X_train_final.to_parquet(os.path.join(OUT_DIR, 'X_TRAIN_FULL_8020.parquet'), index=False)
X_test_final.to_parquet(os.path.join(OUT_DIR, 'X_TEST_FULL_8020.parquet'), index=False)
y_train.to_frame().to_parquet(os.path.join(OUT_DIR, 'Y_TRAIN_FULL_8020.parquet'), index=False)
y_test.to_frame().to_parquet(os.path.join(OUT_DIR, 'Y_TEST_FULL_8020.parquet'), index=False)
print("Full 80/20 Pipeline Complete.")

Full 80/20 Pipeline Complete.
